# ¿A qué famoso me parezco?

In [ ]:
import os
import cv2
import torch
import numpy as np
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from collections import Counter, deque

class FaceEmbeddingModel(nn.Module):
    def __init__(
        self,
        embedding_dim=128
    ):
        super().__init__()

        self.backbone = models.resnet18(
            weights=
            models.ResNet18_Weights.DEFAULT
        )

        num_features = (self.backbone.fc.in_features)

        self.backbone.fc = nn.Sequential(
            nn.Linear(
                num_features,
                embedding_dim
            )
        )

    def forward(self, x):
        embedding = self.backbone(x)

        # Normalización L2
        embedding = F.normalize(
            embedding,
            p=2,
            dim=1
        )

        return embedding

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = FaceEmbeddingModel(128)

checkpoint = torch.load("best_face_embedding_resnet18.pth", map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])

model.to(device)
model.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# Carga del modelo y galería (movemos los embeddings a GPU UNA sola vez)
dnn_model = "deploy.prototxt.txt"
dnn_weights = "res10_300x300_ssd_iter_140000.caffemodel"
net = cv2.dnn.readNetFromCaffe(dnn_model, dnn_weights)

gallery = torch.load("gallery_embeddings.pt", map_location="cpu")
gallery_embeddings = gallery["embeddings"].to(device)
filenames = gallery["filenames"]

model.eval()

# Ventana deslizante: solo recuerda los últimos 20 frames (~0.7 segundos)
votation = deque(maxlen=20)


def get_celeba_crop(frame, box):
    H, W = frame.shape[:2]
    x1, y1, x2, y2 = box

    w = x2 - x1
    h = y2 - y1
    cx = x1 + w / 2.0
    cy = y1 + h / 2.0

    new_h = h * 1.65
    new_w = new_h * (178.0 / 218.0)

    cy_shifted = cy - (h * 0.08)

    nx1 = int(max(0, cx - new_w / 2.0))
    ny1 = int(max(0, cy_shifted - new_h / 2.0))
    nx2 = int(min(W, cx + new_w / 2.0))
    ny2 = int(min(H, cy_shifted + new_h / 2.0))

    return frame[ny1:ny2, nx1:nx2], (nx1, ny1, nx2, ny2)


def get_most_similar_famous(face):
    face_rgb = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
    face_image = Image.fromarray(face_rgb)

    face_tensor = transform(face_image).unsqueeze(0).to(device)

    with torch.no_grad():
        query_embedding = model(face_tensor)
        query_embedding = F.normalize(query_embedding, p=2, dim=1)

    similarities = (query_embedding @ gallery_embeddings.T).squeeze(0)

    _, top_k_indices = torch.topk(similarities, k=3)
    best_index = top_k_indices[0].item()

    route = "." + filenames[best_index]
    votation.append(route)

    most_common_route, _ = Counter(votation).most_common(1)[0]
    return most_common_route

video_capture = cv2.VideoCapture(0)

while True:
    ret, frame = video_capture.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    H, W = frame.shape[:2]

    blob = cv2.dnn.blobFromImage(frame, 1.0, (300, 300), [104.0, 117.0, 123.0], False, False)
    net.setInput(blob)
    detections = net.forward()

    best_idx = np.argmax(detections[0, 0, :, 2])
    max_confidence = detections[0, 0, best_idx, 2]

    if max_confidence > 0.75:
        box = detections[0, 0, best_idx, 3:7] * np.array([W, H, W, H])
        face_crop, (nx1, ny1, nx2, ny2) = get_celeba_crop(frame, box)

        cv2.rectangle(frame, (nx1, ny1), (nx2, ny2), (0, 255, 0), 2)

        if face_crop.size > 0:
            found = get_most_similar_famous(face_crop)
        if found is not None:
            overlay = cv2.imread(found)
            if overlay is not None:
                h, w = overlay.shape[:2]
                x = max(0, min((W - w) // 2, W - w))
                y = max(0, min(20, H - h))
                frame[y : y + h, x : x + w] = overlay
    else:
        votation.clear()

    cv2.putText(frame, "Presiona 'ESC' para salir",
                (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 255), 2)
    cv2.imshow('Face Detection', frame)

    if (cv2.waitKey(1) & 0xFF) == 27:
        break

video_capture.release()
cv2.destroyAllWindows()